# Tiling Fix and Correct Implementation for Anomalib

> Complete guide to fixing tiling issues and implementing tiling correctly with anomalib v2.0+

This notebook addresses tiling issues in anomalib and provides the correct implementation for both anomalib v1.x and v2.x.

## Overview

### The Tiling Problem in Anomalib < 2.0

**Issue**: Tiling was not working correctly in anomalib versions < 2.0, particularly for STFPM and Reverse Distillation models.

**Root Cause**: Models expected the tiler to be initialized during class construction, but it was actually initialized via a callback later. This timing mismatch caused incorrect dimension handling.

**Fix**: The issue was resolved in PR #1319 (September 2023) and properly implemented in anomalib v2.0.0+

### Key Findings

1. **Anomalib < 2.0**: Had tiling bugs, particularly in versions around 0.7.0
2. **Anomalib >= 2.0**: Tiling works correctly with proper `TilerConfigurationCallback`
3. **Your current version**: You're using `anomalib<2`, which may have the bug

### Recommendation

**Upgrade to anomalib >= 2.0** for reliable tiling support. This notebook shows you how.

In [ ]:
#| default_exp tiling.implementation

In [ ]:
#| export
import sys
import warnings
from pathlib import Path
from typing import Tuple, Optional, Dict, Any, List

# Core libraries
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Suppress warnings
warnings.filterwarnings('ignore')

## Step 1: Check Your Current Anomalib Version

In [ ]:
import anomalib

print(f"Current anomalib version: {anomalib.__version__}")
print(f"\nVersion Analysis:")

version_parts = anomalib.__version__.split('.')
major_version = int(version_parts[0])

if major_version < 2:
    print(f"⚠️  You are using anomalib v{anomalib.__version__}")
    print(f"   Tiling may not work correctly in this version.")
    print(f"   Recommendation: Upgrade to anomalib >= 2.0")
    print(f"\n   Known issues in v1.x:")
    print(f"   • Tiling fails for STFPM and Reverse Distillation models")
    print(f"   • Tiler initialization timing issues")
    print(f"   • Incorrect dimension handling during untiling")
else:
    print(f"✅ You are using anomalib v{anomalib.__version__}")
    print(f"   Tiling should work correctly in this version.")

## Step 2: Correct Tiling Implementation for Anomalib v2.0+

Here's the **correct** way to use tiling in anomalib v2.0 and later.

In [ ]:
#| export
# Correct imports for anomalib v2.0+
from anomalib import TaskType
from anomalib.data.image.folder import Folder
from anomalib.engine import Engine
from anomalib.models import Padim, Patchcore
from anomalib.callbacks import TilerConfigurationCallback  # Note: TilerConfigurationCallback, not TilingConfigurationCallback
from anomalib.deploy import ExportType

print("✅ Imports successful")
print(f"   Using: anomalib.callbacks.TilerConfigurationCallback")
print(f"   NOT: anomalib.callbacks.TilingConfigurationCallback (this is incorrect)")

## Step 3: Complete Working Example with Tiling

### Example 1: Training with Tiling (Recommended for Large Images)

In [ ]:
#| eval: false
def train_with_tiling(
    data_root: Path,
    normal_dir: str = "good",
    abnormal_dir: str = "bad",
    class_name: str = "defect",
    tile_size: Tuple[int, int] = (256, 256),
    stride: int = 128,
    image_size: Tuple[int, int] = (512, 512),  # Original large image size
    max_epochs: int = 50,
    model_type: str = "padim"
) -> Dict[str, Any]:
    """
    Train an anomaly detection model with tiling for large images.
    
    Args:
        data_root: Root directory containing training data
        normal_dir: Directory name for normal images
        abnormal_dir: Directory name for abnormal images
        class_name: Name of the defect class
        tile_size: Size of each tile (height, width)
        stride: Step size between tiles (controls overlap)
        image_size: Size of input images (should match your actual image size)
        max_epochs: Number of training epochs
        model_type: Model to use ("padim" or "patchcore")
        
    Returns:
        Dictionary with training results
    """
    
    print(f"🚀 Training with Tiling Configuration")
    print(f"   Data: {data_root}")
    print(f"   Image size: {image_size}")
    print(f"   Tile size: {tile_size}")
    print(f"   Stride: {stride}")
    print(f"   Model: {model_type}")
    print()
    
    # 1. Create datamodule
    datamodule = Folder(
        name=class_name,
        root=data_root,
        normal_dir=normal_dir,
        abnormal_dir=abnormal_dir,
        task=TaskType.CLASSIFICATION,
        image_size=image_size,  # Use full image size
        train_batch_size=16,
        eval_batch_size=16,
        num_workers=0  # Use 0 for HPC/NFS environments
    )
    
    # 2. Create model
    if model_type.lower() == "padim":
        model = Padim()
    elif model_type.lower() == "patchcore":
        model = Patchcore()
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # 3. **CRITICAL**: Create TilerConfigurationCallback with correct parameters
    tiler_callback = TilerConfigurationCallback(
        enable=True,              # Must set to True to enable tiling
        tile_size=tile_size,      # Size of each tile
        stride=stride,            # Overlap between tiles
        remove_border_count=0,    # Remove border artifacts (optional)
        mode="padding"            # How to handle edge tiles: "padding" or "interpolation"
    )
    
    print(f"✅ Created TilerConfigurationCallback:")
    print(f"   enable=True")
    print(f"   tile_size={tile_size}")
    print(f"   stride={stride}")
    print()
    
    # 4. Create engine with tiler callback
    engine = Engine(
        task=TaskType.CLASSIFICATION,
        callbacks=[tiler_callback],  # Pass tiler callback here
        max_epochs=max_epochs,
        accelerator="auto",
        devices="auto"
    )
    
    # 5. Train the model
    print(f"🏋️  Starting training with tiling...")
    engine.fit(model=model, datamodule=datamodule)
    
    # 6. Test the model
    print(f"\n🧪 Testing model...")
    test_results = engine.test(model=model, datamodule=datamodule)
    
    # 7. Export model
    export_path = Path(f"./models/{class_name}_with_tiling")
    export_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\n💾 Exporting model...")
    torch_export = engine.export(
        model=model,
        export_type=ExportType.TORCH,
        export_root=export_path
    )
    
    results = {
        'success': True,
        'test_results': test_results[0] if test_results else None,
        'export_path': str(torch_export),
        'tiling_config': {
            'tile_size': tile_size,
            'stride': stride,
            'image_size': image_size
        }
    }
    
    print(f"\n✅ Training completed!")
    print(f"   Model exported to: {torch_export}")
    print(f"   Test results: {test_results[0] if test_results else 'N/A'}")
    
    return results

### Example 2: Inference with Tiling

After training, you can use the model for inference on large images.

In [ ]:
#| eval: false
def inference_with_tiling(
    model_path: Path,
    image_path: Path,
    tile_size: Tuple[int, int] = (256, 256),
    stride: int = 128
) -> Dict[str, Any]:
    """
    Run inference on a large image using tiling.
    
    Args:
        model_path: Path to exported torch model
        image_path: Path to image to analyze
        tile_size: Size of tiles used during training
        stride: Stride used during training
        
    Returns:
        Dictionary with prediction results
    """
    from anomalib.deploy import TorchInferencer
    from anomalib.data.utils.tiler import Tiler
    
    print(f"🔍 Running inference with tiling")
    print(f"   Model: {model_path}")
    print(f"   Image: {image_path}")
    print(f"   Tile size: {tile_size}")
    print(f"   Stride: {stride}")
    
    # 1. Load the trained model
    inferencer = TorchInferencer(
        path=model_path,
        device="auto"
    )
    
    # 2. Load and prepare image
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    
    print(f"   Image shape: {image_np.shape}")
    
    # 3. Create tiler for processing large image
    tiler = Tiler(
        tile_size=tile_size,
        stride=stride
    )
    
    # 4. Run inference
    # Note: TorchInferencer will automatically handle tiling if the model was trained with it
    result = inferencer.predict(image=image_np)
    
    print(f"\n✅ Inference completed")
    print(f"   Prediction: {'ANOMALY' if result.pred_label else 'NORMAL'}")
    print(f"   Score: {result.pred_score:.4f}")
    print(f"   Anomaly map shape: {result.anomaly_map.shape if hasattr(result, 'anomaly_map') else 'N/A'}")
    
    return {
        'pred_label': result.pred_label,
        'pred_score': float(result.pred_score),
        'anomaly_map': result.anomaly_map if hasattr(result, 'anomaly_map') else None,
        'image_shape': image_np.shape
    }

### Example 3: Using Tiler Class Directly

For advanced use cases, you can use the `Tiler` class directly.

In [ ]:
#| eval: false
def demonstrate_tiler_directly():
    """
    Demonstrate using the Tiler class directly for custom tiling operations.
    """
    from anomalib.data.utils.tiler import Tiler
    
    print("🔬 Demonstrating Direct Tiler Usage\n")
    
    # 1. Create a sample large image (512x512)
    large_image = torch.rand(1, 3, 512, 512)
    print(f"Original image shape: {large_image.shape}")
    
    # 2. Create tiler with 256x256 tiles and 128 stride (50% overlap)
    tiler = Tiler(
        tile_size=(256, 256),
        stride=128,
        remove_border_count=0,
        mode="padding"
    )
    
    print(f"\nTiler configuration:")
    print(f"   tile_size: {tiler.tile_size}")
    print(f"   stride: {tiler.stride}")
    print(f"   mode: {tiler.mode}")
    
    # 3. Tile the image
    tiles = tiler.tile(large_image)
    print(f"\nTiling result:")
    print(f"   Number of tiles: {tiles.shape[0]}")
    print(f"   Each tile shape: {tiles.shape[1:]}")
    
    # 4. Process tiles (simulate anomaly detection)
    print(f"\nProcessing tiles...")
    processed_tiles = tiles * 0.9  # Simulate some processing
    
    # 5. Untile to reconstruct the image
    reconstructed = tiler.untile(processed_tiles)
    print(f"\nReconstructed image shape: {reconstructed.shape}")
    
    # 6. Verify reconstruction
    reconstruction_error = torch.abs(large_image - reconstructed).mean()
    print(f"\n✅ Tiling/Untiling Test:")
    print(f"   Reconstruction error: {reconstruction_error:.6f}")
    print(f"   Status: {'PASS' if reconstruction_error < 0.1 else 'FAIL'}")
    
    return {
        'original_shape': large_image.shape,
        'num_tiles': tiles.shape[0],
        'tile_shape': tiles.shape[1:],
        'reconstructed_shape': reconstructed.shape,
        'reconstruction_error': float(reconstruction_error)
    }

# Run the demonstration
demo_results = demonstrate_tiler_directly()
print(f"\nDemo completed successfully!")

## Step 4: Migration from Anomalib < 2.0 to >= 2.0

If you need to upgrade, here's what changed:

In [ ]:
print("📋 Migration Guide: Anomalib v1.x → v2.x")
print("=" * 60)

migration_guide = """
### Key Changes in Tiling:

1. Import Path (UNCHANGED):
   ✅ v1.x: from anomalib.callbacks import TilerConfigurationCallback
   ✅ v2.x: from anomalib.callbacks import TilerConfigurationCallback
   
2. Callback Parameters (SLIGHTLY CHANGED):
   v1.x:
       TilerConfigurationCallback(tile_size=(256, 256), stride=128)
   
   v2.x (recommended):
       TilerConfigurationCallback(
           enable=True,              # ← NEW: Must explicitly enable
           tile_size=(256, 256),
           stride=128,
           remove_border_count=0,    # ← NEW: Optional parameter
           mode="padding"            # ← NEW: "padding" or "interpolation"
       )

3. Engine Creation (CHANGED):
   v1.x:
       from anomalib.engine import Engine
       engine = Engine(
           accelerator="auto",
           callbacks=[tiler_callback],
           ...
       )
   
   v2.x (SAME, but with task parameter):
       from anomalib.engine import Engine
       from anomalib import TaskType
       engine = Engine(
           task=TaskType.CLASSIFICATION,  # ← NEW: Must specify task
           accelerator="auto",
           callbacks=[tiler_callback],
           ...
       )

4. Tiler Class Direct Usage (MOSTLY UNCHANGED):
   Both versions:
       from anomalib.data.utils.tiler import Tiler
       tiler = Tiler(tile_size=256, stride=128)
       tiles = tiler.tile(image)
       reconstructed = tiler.untile(tiles)

### Upgrade Command:
   
   # Upgrade to anomalib v2.x
   pip install "anomalib>=2.0"
   
   # Or update pyproject.toml:
   anomalib>=2.0,<3
"""

print(migration_guide)

## Step 5: Common Issues and Solutions

In [ ]:
print("🔧 Common Tiling Issues and Solutions")
print("=" * 60)

troubleshooting_guide = """
### Issue 1: Tiling Not Working (Images Not Being Tiled)
   
   Symptoms:
   - Model processes full images instead of tiles
   - Out of memory errors with large images
   
   Solutions:
   ✅ Make sure enable=True in TilerConfigurationCallback
   ✅ Verify callback is passed to Engine: Engine(callbacks=[tiler_callback])
   ✅ Check anomalib version (should be >= 2.0 for reliability)

### Issue 2: Dimension Mismatch Errors
   
   Symptoms:
   - "RuntimeError: Dimension out of range"
   - Incorrect anomaly map sizes
   
   Solutions:
   ✅ Ensure tile_size and stride are compatible with image_size
   ✅ Use mode="padding" to handle edge cases
   ✅ Upgrade to anomalib >= 2.0 (fixes tiler initialization bug)

### Issue 3: Poor Anomaly Detection Performance
   
   Symptoms:
   - Lower accuracy with tiling vs without
   - Artifacts at tile boundaries
   
   Solutions:
   ✅ Use adequate overlap: stride = tile_size // 2 (50% overlap)
   ✅ Set remove_border_count=8 to eliminate edge artifacts
   ✅ Ensure tile_size is large enough to capture defect features (≥256)

### Issue 4: Wrong Import Name
   
   Symptoms:
   - ImportError: cannot import name 'TilingConfigurationCallback'
   
   Solution:
   ❌ WRONG: from anomalib.callbacks import TilingConfigurationCallback
   ✅ CORRECT: from anomalib.callbacks import TilerConfigurationCallback
   
   Note the difference: Tiler vs Tiling (use Tiler)

### Issue 5: Tiling in Inference Not Working
   
   Symptoms:
   - Model trained with tiling but inference fails
   - Different results between training and inference
   
   Solutions:
   ✅ Use same tile_size and stride for inference as training
   ✅ Use TorchInferencer.predict() - it handles tiling automatically
   ✅ Verify exported model includes tiler configuration
"""

print(troubleshooting_guide)

## Step 6: Best Practices and Recommendations

In [ ]:
print("💡 Best Practices for Tiling in Anomalib")
print("=" * 60)

best_practices = """
### When to Use Tiling:

✅ Use tiling when:
   • Images are larger than 512x512 pixels
   • GPU memory is limited
   • Defects are small relative to image size
   • You need to detect anomalies in specific regions

❌ Avoid tiling when:
   • Images are already small (< 512x512)
   • Anomalies span large areas
   • Sufficient GPU memory available
   • Training time is critical (tiling adds overhead)

### Recommended Parameters:

1. tile_size:
   • Minimum: 256x256 (for most models)
   • Recommended: 256x256 or 512x512
   • Must be compatible with model architecture

2. stride:
   • For best accuracy: stride = tile_size // 2 (50% overlap)
   • For faster processing: stride = tile_size (no overlap)
   • For critical applications: stride = tile_size // 4 (75% overlap)

3. remove_border_count:
   • Default: 0 (no border removal)
   • For cleaner results: 8-16 pixels
   • Reduces edge artifacts from tiling

4. mode:
   • "padding": Pads images to fit tile grid (recommended)
   • "interpolation": Resizes images to fit (may distort features)

### Example Configuration for Different Image Sizes:

Small images (512x512):
   TilerConfigurationCallback(
       enable=True,
       tile_size=(256, 256),
       stride=128,  # 50% overlap
       mode="padding"
   )

Medium images (1024x1024):
   TilerConfigurationCallback(
       enable=True,
       tile_size=(512, 512),
       stride=256,  # 50% overlap
       mode="padding"
   )

Large images (2048x2048 or larger):
   TilerConfigurationCallback(
       enable=True,
       tile_size=(512, 512),
       stride=128,  # 75% overlap for critical defects
       remove_border_count=16,
       mode="padding"
   )

### Performance Optimization:

• Larger tiles = fewer tiles = faster training (but needs more memory)
• Smaller stride = more overlap = better accuracy (but slower)
• Balance tile_size and stride based on your:
  - Available GPU memory
  - Defect size
  - Accuracy requirements
  - Training time constraints
"""

print(best_practices)

## Step 7: Complete Working Example

Here's a complete example you can run:

In [ ]:
#| eval: false

# COMPLETE WORKING EXAMPLE
# This example shows the correct way to use tiling in anomalib v2.0+

def complete_tiling_example(
    data_root: str = "/path/to/your/data",
    output_dir: str = "./tiling_results"
):
    """
    Complete end-to-end example of training and inference with tiling.
    
    Directory structure expected:
        data_root/
            good/       # Normal images
                img1.png
                img2.png
                ...
            bad/        # Defective images (optional for training)
                defect1.png
                defect2.png
                ...
    """
    
    from pathlib import Path
    from anomalib import TaskType
    from anomalib.data.image.folder import Folder
    from anomalib.engine import Engine
    from anomalib.models import Padim
    from anomalib.callbacks import TilerConfigurationCallback
    from anomalib.deploy import ExportType, TorchInferencer
    
    print("🚀 COMPLETE TILING EXAMPLE")
    print("=" * 60)
    
    data_root = Path(data_root)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Configuration
    config = {
        'image_size': (1024, 1024),  # Your actual image size
        'tile_size': (512, 512),     # Size of each tile
        'stride': 256,                # 50% overlap
        'max_epochs': 50,
        'class_name': 'defect_with_tiling'
    }
    
    print(f"\n📋 Configuration:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    # ========================================
    # STEP 1: Prepare Data
    # ========================================
    print(f"\n📁 STEP 1: Preparing data...")
    
    datamodule = Folder(
        name=config['class_name'],
        root=data_root,
        normal_dir="good",
        abnormal_dir="bad",
        task=TaskType.CLASSIFICATION,
        image_size=config['image_size'],
        train_batch_size=8,
        eval_batch_size=8,
        num_workers=0
    )
    
    print(f"   ✅ Data module created")
    
    # ========================================
    # STEP 2: Create Model
    # ========================================
    print(f"\n🧠 STEP 2: Creating model...")
    
    model = Padim(
        backbone="resnet18",
        layers=["layer1", "layer2", "layer3"]
    )
    
    print(f"   ✅ Model created: Padim with resnet18")
    
    # ========================================
    # STEP 3: Configure Tiling (CRITICAL!)
    # ========================================
    print(f"\n🔲 STEP 3: Configuring tiling...")
    
    tiler_callback = TilerConfigurationCallback(
        enable=True,                          # Must be True!
        tile_size=config['tile_size'],
        stride=config['stride'],
        remove_border_count=8,                # Remove edge artifacts
        mode="padding"                        # Pad images to fit
    )
    
    print(f"   ✅ Tiler configured:")
    print(f"      • tile_size: {config['tile_size']}")
    print(f"      • stride: {config['stride']}")
    print(f"      • overlap: {(config['tile_size'][0] - config['stride']) / config['tile_size'][0] * 100:.0f}%")
    
    # ========================================
    # STEP 4: Create Engine with Tiling
    # ========================================
    print(f"\n⚙️  STEP 4: Creating engine...")
    
    engine = Engine(
        task=TaskType.CLASSIFICATION,
        callbacks=[tiler_callback],           # Pass tiler callback
        max_epochs=config['max_epochs'],
        accelerator="auto",
        devices="auto"
    )
    
    print(f"   ✅ Engine created with tiling callback")
    
    # ========================================
    # STEP 5: Train
    # ========================================
    print(f"\n🏋️  STEP 5: Training model...")
    print(f"   This may take a while...\n")
    
    engine.fit(model=model, datamodule=datamodule)
    
    print(f"\n   ✅ Training completed!")
    
    # ========================================
    # STEP 6: Test
    # ========================================
    print(f"\n🧪 STEP 6: Testing model...")
    
    test_results = engine.test(model=model, datamodule=datamodule)
    
    if test_results:
        print(f"   ✅ Test results:")
        for key, value in test_results[0].items():
            print(f"      • {key}: {value}")
    
    # ========================================
    # STEP 7: Export Model
    # ========================================
    print(f"\n💾 STEP 7: Exporting model...")
    
    export_path = engine.export(
        model=model,
        export_type=ExportType.TORCH,
        export_root=output_dir / "exported_model"
    )
    
    print(f"   ✅ Model exported to: {export_path}")
    
    # ========================================
    # STEP 8: Inference Example
    # ========================================
    print(f"\n🔍 STEP 8: Running inference example...")
    
    # Load inferencer
    inferencer = TorchInferencer(
        path=export_path,
        device="auto"
    )
    
    # Find a test image
    test_images = list((data_root / "good").glob("*.png"))[:1]
    if test_images:
        test_image_path = test_images[0]
        
        # Load image
        from PIL import Image
        import numpy as np
        
        image = np.array(Image.open(test_image_path))
        
        # Run prediction (automatically handles tiling)
        result = inferencer.predict(image=image)
        
        print(f"   ✅ Inference on {test_image_path.name}:")
        print(f"      • Prediction: {'ANOMALY' if result.pred_label else 'NORMAL'}")
        print(f"      • Score: {result.pred_score:.4f}")
    
    # ========================================
    # Summary
    # ========================================
    print(f"\n" + "=" * 60)
    print(f"✅ COMPLETE! All steps finished successfully.")
    print(f"\n📊 Summary:")
    print(f"   • Model: Padim with tiling")
    print(f"   • Image size: {config['image_size']}")
    print(f"   • Tile size: {config['tile_size']}")
    print(f"   • Stride: {config['stride']}")
    print(f"   • Exported to: {export_path}")
    print(f"\n🎉 You can now use the trained model for inference!")
    print("=" * 60)
    
    return {
        'model_path': export_path,
        'config': config,
        'test_results': test_results[0] if test_results else None
    }


# To run this example:
# results = complete_tiling_example(
#     data_root="/path/to/your/data",
#     output_dir="./my_tiling_results"
# )

## Summary

### Key Takeaways:

1. **Anomalib < 2.0 has tiling bugs** - Upgrade to v2.0+ for reliable tiling
2. **Correct import**: `from anomalib.callbacks import TilerConfigurationCallback` (not TilingConfigurationCallback)
3. **Must set enable=True** in TilerConfigurationCallback
4. **Pass callback to Engine**: `Engine(callbacks=[tiler_callback])`
5. **Use adequate overlap**: stride = tile_size // 2 for best results

### References:

- [Anomalib v2.0 Tiling Documentation](https://anomalib.readthedocs.io/en/v2.0.0/markdown/guides/how_to/data/input_tiling.html)
- [GitHub Issue #1318: Tiling Bug](https://github.com/openvinotoolkit/anomalib/issues/1318)
- [Anomalib Callbacks Documentation](https://anomalib.readthedocs.io/en/v1.1.1/markdown/guides/reference/callbacks/)

### Next Steps:

1. Check your anomalib version
2. If < 2.0, upgrade: `pip install "anomalib>=2.0"`
3. Update your code using the examples in this notebook
4. Test tiling with your data
5. Tune tile_size and stride for your use case